In [1]:
# =============================================================================
# NOTEBOOK: 07_cross_domain_demonstration.ipynb
# Automating Interview-Based Generative-AI ROI Measurement
#   — Cross-Domain Transferability Demonstration
#
# GOAL: Show that the SAME pipeline (same prompts, same code, same design
# principles) runs on interviews from ENTIRELY DIFFERENT domains with no
# domain-specific modification. The only thing that changes is the input file.
# This operationalizes the paper's central transferability claim.
#
# Method: run Agent 1 (extraction + grading) — the pipeline's entry stage — on
# three short synthetic interviews from unrelated domains, and compare the
# structure of the outputs. We keep it to Agent 1 for a compact, cheap demo;
# the full pipeline would follow identically.
#
# Domains demonstrated:
#   (A) Hospital billing / revenue-cycle back office
#   (B) Law-firm contract review
#   (C) Manufacturing monthly settlement (the 기획서's original example domain)
#
# All content and code are in English for journal submission.
# =============================================================================


# %%
# =============================================================================
# Cell 1. Bootstrap (reuse nb 00 foundation, self-contained)
# =============================================================================
import os, re, json, time, hashlib, warnings
from pathlib import Path
from typing import Any
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
ARTIFACTS = ROOT / "artifacts"
INFER = ARTIFACTS / "inference"
TAB = ARTIFACTS / "tables"
CACHE = ARTIFACTS / "llm_cache"
DATA_RAW = ROOT / "data" / "raw"
for p in (INFER, TAB, CACHE, DATA_RAW):
    p.mkdir(parents=True, exist_ok=True)

def rel(p):
    try: return str(Path(p).resolve().relative_to(ROOT.resolve()))
    except ValueError: return Path(p).name

manifest = json.loads((ARTIFACTS / "run_manifest.json").read_text(encoding="utf-8"))
MODELS = manifest["models"]; DEFAULT_TIER = manifest["default_tier"]
AUTO_GRADES = manifest["auto_grades"]; MISSING = manifest["missing_sentinel"]

load_dotenv(ROOT / ".env")
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Minimal llm_call (JSON, cached) — identical behavior to nb 00.
def _safe_json(t):
    if not t: return None
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", t.strip(), flags=re.S).strip()
    try: return json.loads(t)
    except Exception:
        for o, c in (("{", "}"), ("[", "]")):
            i, j = t.find(o), t.rfind(c)
            if 0 <= i < j:
                try: return json.loads(t[i:j+1])
                except Exception: pass
    return None

def llm_call(system, user, tier=DEFAULT_TIER, temperature=0.0, tag=""):
    model = MODELS[tier]["name"]
    key = hashlib.sha256(json.dumps({"s": system, "u": user, "m": model,
          "t": temperature}, sort_keys=True).encode()).hexdigest()[:24]
    cf = CACHE / f"{key}.json"
    if cf.exists():
        return json.loads(cf.read_text(encoding="utf-8"))
    resp = client.chat.completions.create(
        model=model, temperature=temperature,
        response_format={"type": "json_object"},
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": user}])
    out = {"json": _safe_json(resp.choices[0].message.content or "")}
    cf.write_text(json.dumps(out, ensure_ascii=False), encoding="utf-8")
    return out

print("[INFO] Cross-domain demo bootstrap ready.")


# %%
# =============================================================================
# Cell 2. Three short interviews from unrelated domains (synthetic, English)
#
# These are deliberately compact. The point is NOT depth but that the SAME
# extraction prompt handles all three with no change.
# =============================================================================
INTERVIEWS = {
    "hospital_billing": """\
[Interview — Hospital Revenue-Cycle Team]
Q. How does a claim get processed?
A. A coder reads the physician's clinical notes and assigns diagnosis and
procedure codes. Then a biller checks the codes against payer rules and builds
the claim in the billing system. If required fields are missing, the claim is
held and a query is sent back to the physician. Once complete, the claim is
submitted electronically to the insurer. When a denial comes back, a specialist
reads the denial reason, gathers supporting documents, and writes an appeal
letter. Posting the payment to the patient ledger is done by hand at month end.
""",
    "law_contract_review": """\
[Interview — Corporate Legal, Contract Review]
Q. Walk me through a contract review.
A. A paralegal intakes the counterparty's draft and logs it in the matter
system. An associate reads each clause and compares it against our standard
playbook, flagging deviations on indemnity, liability caps, and termination.
For non-standard clauses the associate drafts proposed redlines. A partner
reviews high-risk clauses and signs off. The negotiated changes are emailed to
the counterparty, and the final executed copy is filed in the document
repository. Tracking key dates like renewal and expiry is done manually in a
spreadsheet.
""",
    "manufacturing_settlement": """\
[Interview — Manufacturing Finance, Monthly Settlement]
Q. How does monthly settlement work?
A. An analyst pulls production volumes from the MES and material costs from the
ERP. They reconcile the two against work orders, and where quantities do not
match they investigate the variance with the plant. Overhead is allocated by a
fixed rule. The analyst prepares the cost-of-goods journal entry and a manager
reviews and approves it. The closing report is generated from a template and
distributed to the plant controllers by email.
""",
}
for k, v in INTERVIEWS.items():
    print(f"[INFO] {k}: {len(v)} chars")


# %%
# =============================================================================
# Cell 3. The SAME Agent-1 prompt as nb 01 (verbatim, domain-agnostic)
# =============================================================================
AGENT1_SYSTEM = f"""\
You are Agent 1 (Task Extraction & Automatability Classification) in a pipeline
that estimates the ROI of adopting generative AI for knowledge work. You analyze
an operational interview transcript from ANY business domain and identify the
discrete, repeatable work units it describes.

Reason step by step before the final answer, then for each unit decide the
actor and system IF stated (else "{MISSING}"), grade AI-automatability as one of
"full" ({AUTO_GRADES['full']}), "partial" ({AUTO_GRADES['partial']}), or
"manual" ({AUTO_GRADES['manual']}), and give a one-sentence rationale.

Return ONLY JSON:
{{ "work_units": [ {{"id":"w1","name":"...","actor":"...","system":"...",
"auto_grade":"full|partial|manual","auto_rationale":"...","evidence":"..."}} ] }}
"""

def extract(interview_text):
    r = llm_call(AGENT1_SYSTEM,
                 f"--- TRANSCRIPT ---\n{interview_text}\n--- END ---",
                 tier=DEFAULT_TIER, temperature=0.0, tag="crossdomain_a1")
    return (r["json"] or {}).get("work_units", [])

print("[INFO] Using the identical Agent-1 prompt across all domains.")


# %%
# =============================================================================
# Cell 4. Run the SAME pipeline stage on all three domains
# =============================================================================
results = {}
for domain, text in INTERVIEWS.items():
    units = extract(text)
    results[domain] = units
    dist = {}
    for u in units:
        g = u.get("auto_grade", MISSING)
        dist[g] = dist.get(g, 0) + 1
    print(f"[INFO] {domain:26s}: {len(units):2d} units  grades={dist}")


# %%
# =============================================================================
# Cell 5. Comparison table — same structure emerges in every domain
# =============================================================================
rows = []
for domain, units in results.items():
    dist = {"full": 0, "partial": 0, "manual": 0}
    for u in units:
        g = u.get("auto_grade")
        if g in dist:
            dist[g] += 1
    rows.append({
        "domain": domain,
        "work_units": len(units),
        "full": dist["full"], "partial": dist["partial"], "manual": dist["manual"],
        "%automatable(full+partial)":
            round((dist["full"] + dist["partial"]) / max(len(units), 1) * 100, 1),
    })
cmp_df = pd.DataFrame(rows)
print(cmp_df.to_string(index=False))
out_csv = TAB / "cross_domain_comparison.csv"
cmp_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"\n[INFO] Comparison table -> {rel(out_csv)}")

# Persist the per-domain extractions for inspection / appendix.
out_json = INFER / "cross_domain_extractions.json"
out_json.write_text(json.dumps(results, ensure_ascii=False, indent=2),
                    encoding="utf-8")
print(f"[INFO] Extractions -> {rel(out_json)}")


# %%
# =============================================================================
# Cell 6. Takeaway (printed) — the transferability claim, evidenced
# =============================================================================
print("""
[TRANSFERABILITY EVIDENCE]
The identical Agent-1 prompt and code produced a well-formed, graded work-unit
list for three unrelated domains (hospital billing, legal contract review,
manufacturing settlement) with NO domain-specific changes. Only the input file
differed. Each domain yields the same output schema and the same three-grade
automatability structure, confirming that domain knowledge resides entirely in
the input — the design's core transferability property. The full pipeline
(Agents 1.5 → 2 → 3 and the five-axis validation) would apply identically.
""")

[INFO] Cross-domain demo bootstrap ready.
[INFO] hospital_billing: 620 chars
[INFO] law_contract_review: 627 chars
[INFO] manufacturing_settlement: 532 chars
[INFO] Using the identical Agent-1 prompt across all domains.
[INFO] hospital_billing          :  6 units  grades={'partial': 3, 'full': 2, 'manual': 1}
[INFO] law_contract_review       :  7 units  grades={'full': 2, 'partial': 3, 'manual': 2}
[INFO] manufacturing_settlement  :  7 units  grades={'partial': 4, 'full': 2, 'manual': 1}
                  domain  work_units  full  partial  manual  %automatable(full+partial)
        hospital_billing           6     2        3       1                        83.3
     law_contract_review           7     2        3       2                        71.4
manufacturing_settlement           7     2        4       1                        85.7

[INFO] Comparison table -> artifacts\tables\cross_domain_comparison.csv
[INFO] Extractions -> artifacts\inference\cross_domain_extractions.json

[TRANSFER

In [2]:
# %%
# =============================================================================
# Cell 7. Appendix export — full interviews + per-domain extraction tables
#
# Emits journal-appendix-ready assets from the cross-domain run:
#   (1) each domain's full input interview as a text file,
#   (2) each domain's extracted work units as a CSV table,
#   (3) a combined long-format CSV across all domains,
#   (4) a LaTeX table per domain for direct paste into the appendix.
# No new API calls — this only serializes what Cell 4 already produced.
# =============================================================================
APX = ARTIFACTS / "appendix"
APX.mkdir(parents=True, exist_ok=True)

DOMAIN_TITLES = {
    "hospital_billing": "Hospital Revenue-Cycle / Billing",
    "law_contract_review": "Corporate Legal — Contract Review",
    "manufacturing_settlement": "Manufacturing Finance — Monthly Settlement",
}
COLS = ["id", "name", "actor", "system", "auto_grade", "auto_rationale"]

def esc_latex(s: str) -> str:
    """Escape the characters that would break a LaTeX table cell."""
    s = str(s or "")
    for a, b in [("\\", r"\textbackslash{}"), ("&", r"\&"), ("%", r"\%"),
                 ("$", r"\$"), ("#", r"\#"), ("_", r"\_"), ("{", r"\{"),
                 ("}", r"\}"), ("~", r"\textasciitilde{}"),
                 ("^", r"\textasciicircum{}")]:
        s = s.replace(a, b)
    return s

combined_rows = []
for domain, units in results.items():
    title = DOMAIN_TITLES.get(domain, domain)

    # (1) full interview text
    (APX / f"interview_{domain}.txt").write_text(
        INTERVIEWS[domain].strip(), encoding="utf-8")

    # (2) per-domain CSV
    df = pd.DataFrame([{c: u.get(c, "") for c in COLS} for u in units])
    df.to_csv(APX / f"units_{domain}.csv", index=False, encoding="utf-8-sig")

    # accumulate for the combined table
    for u in units:
        combined_rows.append({"domain": title, **{c: u.get(c, "") for c in COLS}})

    # (4) per-domain LaTeX table (booktabs style)
    lines = [
        r"\begin{table}[htbp]\centering",
        r"\small",
        rf"\caption{{Extracted work units — {esc_latex(title)}}}",
        rf"\label{{tab:units_{domain}}}",
        r"\begin{tabular}{@{}llll@{}}",
        r"\toprule",
        r"Task & Actor & Grade & Rationale \\",
        r"\midrule",
    ]
    for u in units:
        lines.append(
            f"{esc_latex(u.get('name'))} & {esc_latex(u.get('actor'))} & "
            f"{esc_latex(u.get('auto_grade'))} & "
            f"{esc_latex(u.get('auto_rationale'))} \\\\"
        )
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    (APX / f"units_{domain}.tex").write_text("\n".join(lines), encoding="utf-8")

    print(f"[INFO] {title}: {len(units)} units -> "
          f"units_{domain}.csv / .tex / interview_{domain}.txt")

# (3) combined long-format CSV across all domains
comb = pd.DataFrame(combined_rows)
comb.to_csv(APX / "cross_domain_units_all.csv", index=False, encoding="utf-8-sig")
print(f"[INFO] Combined table ({len(comb)} rows) -> "
      f"{rel(APX / 'cross_domain_units_all.csv')}")
print(f"[INFO] All appendix assets in -> {rel(APX)}")

[INFO] Hospital Revenue-Cycle / Billing: 6 units -> units_hospital_billing.csv / .tex / interview_hospital_billing.txt
[INFO] Corporate Legal — Contract Review: 7 units -> units_law_contract_review.csv / .tex / interview_law_contract_review.txt
[INFO] Manufacturing Finance — Monthly Settlement: 7 units -> units_manufacturing_settlement.csv / .tex / interview_manufacturing_settlement.txt
[INFO] Combined table (20 rows) -> artifacts\appendix\cross_domain_units_all.csv
[INFO] All appendix assets in -> artifacts\appendix


In [3]:
# %%
# =============================================================================
# Cell 8. Appendix export — the illustrative TCB case (same LaTeX treatment)
#
# Unifies the primary illustrative case with the cross-domain appendix tables.
# Loads Agent-1's 46 TCB work units and emits CSV + a longtable LaTeX table
# (longtable handles the multi-page span 46 rows require). Reuses APX and
# esc_latex() defined in the previous cell.
# =============================================================================
tcb_a1 = json.loads(
    (INFER / "agent1_work_units.json").read_text(encoding="utf-8"))
tcb_units = tcb_a1["work_units"]
TCB_TITLE = "Illustrative Case — Technology-Credit Evaluation (TCB)"

# (1) input interview text (copied into the appendix folder for completeness)
tcb_interview = (ROOT / "data" / "raw" / "interview_tcb.txt")
if tcb_interview.exists():
    (APX / "interview_tcb.txt").write_text(
        tcb_interview.read_text(encoding="utf-8").strip(), encoding="utf-8")

# (2) CSV table
tcb_df = pd.DataFrame([{c: u.get(c, "") for c in COLS} for u in tcb_units])
tcb_df.to_csv(APX / "units_tcb.csv", index=False, encoding="utf-8-sig")

# (3) longtable LaTeX (46 rows -> spans pages; longtable repeats the header)
lines = [
    r"\begin{longtable}{@{}p{4.4cm}p{2.6cm}p{1.4cm}p{5.2cm}@{}}",
    rf"\caption{{Extracted work units — {esc_latex(TCB_TITLE)}}}"
    r"\label{tab:units_tcb}\\",
    r"\toprule",
    r"Task & Actor & Grade & Rationale \\",
    r"\midrule",
    r"\endfirsthead",
    r"\multicolumn{4}{c}{\tablename\ \thetable\ -- continued}\\",
    r"\toprule",
    r"Task & Actor & Grade & Rationale \\",
    r"\midrule",
    r"\endhead",
    r"\midrule \multicolumn{4}{r}{continued on next page}\\",
    r"\endfoot",
    r"\bottomrule",
    r"\endlastfoot",
]
for u in tcb_units:
    lines.append(
        f"{esc_latex(u.get('name'))} & {esc_latex(u.get('actor'))} & "
        f"{esc_latex(u.get('auto_grade'))} & "
        f"{esc_latex(u.get('auto_rationale'))} \\\\"
    )
lines.append(r"\end{longtable}")
(APX / "units_tcb.tex").write_text("\n".join(lines), encoding="utf-8")

print(f"[INFO] {TCB_TITLE}: {len(tcb_units)} units -> "
      f"units_tcb.csv / units_tcb.tex / interview_tcb.txt")
print(f"[INFO] Appendix now covers 4 cases (TCB + 3 cross-domain) in {rel(APX)}")

[INFO] Illustrative Case — Technology-Credit Evaluation (TCB): 46 units -> units_tcb.csv / units_tcb.tex / interview_tcb.txt
[INFO] Appendix now covers 4 cases (TCB + 3 cross-domain) in artifacts\appendix


In [6]:
# %%
# =============================================================================
# Cell 9. Apply Agent 1.5 to each cross-domain case -> process_graph_<case>.json
#
# The three cross-domain cases were run through Agent 1 (extraction) only. Here
# we run the SAME Agent-1.5 logic used for TCB (shape typing + edges + AS-IS/
# TO-BE derivation) on each domain, producing a per-domain process graph with
# the IDENTICAL schema as artifacts/inference/process_graph.json. These feed the
# appendix swimlane figures. No pipeline change is domain-specific.
# =============================================================================
PARTIAL_RETAIN = 0.30          # must match the value used elsewhere in the pipeline
AI_LANE = "AI Agent"
AI_LANES = {"system", "gpt", "ai", "ai agent"}
VALID_TYPES = {"start", "task", "decision", "end"}

def is_ai(lane: str) -> bool:
    l = (lane or "").lower()
    return l in AI_LANES or "gpt" in l

# --- Agent-1.5 prompt: assign shape types + edges (verbatim from nb 015) ------
AGENT15_SYSTEM = """\
You are Agent 1.5 (Process-Graph Construction) in an ROI-estimation pipeline.
You receive an ORDERED list of work units already extracted from an operational
interview (any business domain). Turn them into a process graph.

For every unit, assign exactly one flowchart SHAPE TYPE:
  - "start"    : the entry point that triggers the process (usually the first)
  - "end"      : a terminal unit that dispatches/closes the process
  - "decision" : a unit that branches on a condition (yes/no, new/re-run, pass/fail)
  - "task"     : any ordinary activity that is neither start, end, nor a branch

Then declare directed EDGES connecting the units in process-flow order. For a
decision, emit two edges each with a short branch label. Keep it mostly linear;
add branches only where the units imply them. One "start", at least one "end".

Return ONLY JSON:
{ "nodes": [ {"id":"w1","type":"start|task|decision|end"} ],
  "edges": [ {"from":"wX","to":"wY","label":""} ] }
"""

def build_graph_for_case(units: list[dict]) -> dict:
    """Run Agent 1.5 on one domain's units and derive AS-IS/TO-BE graphs."""
    compact = [{"id": u["id"], "name": u["name"],
                "actor": u.get("actor", MISSING),
                "grade": u.get("auto_grade", MISSING)} for u in units]
    r = llm_call(AGENT15_SYSTEM,
                 "Ordered work units:\n" + json.dumps(compact, ensure_ascii=False),
                 tier=DEFAULT_TIER, temperature=0.0, tag="crossdomain_a15")
    g = r["json"] or {}

    id2unit = {u["id"]: u for u in units}
    ordered_ids = [u["id"] for u in units]

    # --- validate/repair node types ---
    type_by_id = {}
    for n in g.get("nodes", []):
        nid = n.get("id"); t = str(n.get("type", "task")).strip().lower()
        if nid in id2unit and t in VALID_TYPES:
            type_by_id[nid] = t
    for u in units:
        type_by_id.setdefault(u["id"], "task")
    if "start" not in type_by_id.values():
        type_by_id[ordered_ids[0]] = "start"
    if "end" not in type_by_id.values():
        type_by_id[ordered_ids[-1]] = "end"

    # --- keep valid edges; add linear backbone if too sparse ---
    edges = [{"from": e["from"], "to": e["to"],
              "label": str(e.get("label", "")).strip()}
             for e in g.get("edges", [])
             if e.get("from") in id2unit and e.get("to") in id2unit]
    if len(edges) < len(units) - 1:
        have = {(e["from"], e["to"]) for e in edges}
        for a, b in zip(ordered_ids, ordered_ids[1:]):
            if (a, b) not in have:
                edges.append({"from": a, "to": b, "label": ""})

    # --- assemble AS-IS nodes ---
    asis_nodes = [{
        "id": u["id"], "name": u["name"], "lane": u.get("actor", MISSING),
        "type": type_by_id[u["id"]], "grade": u.get("auto_grade", MISSING),
        "rationale": u.get("auto_rationale", ""),
    } for u in units]

    # --- derive TO-BE (full->AI, partial->AI draft+human review, manual keep) ---
    tobe_nodes, id_map = [], {}
    for n in asis_nodes:
        g_ = n["grade"]
        base = {"src_id": n["id"], "name": n["name"], "type": n["type"],
                "grade": g_, "rationale": n.get("rationale", "")}
        if g_ == "full":
            nid = n["id"] + "_ai"
            tobe_nodes.append({**base, "id": nid, "lane": AI_LANE,
                               "human_effort_factor": 0.0, "role": "ai"})
            id_map[n["id"]] = [nid]
        elif g_ == "partial":
            ai_id, hr_id = n["id"] + "_ai", n["id"] + "_review"
            tobe_nodes.append({**base, "id": ai_id, "lane": AI_LANE,
                               "name": n["name"] + " (AI draft)",
                               "human_effort_factor": 0.0, "role": "ai"})
            tobe_nodes.append({**base, "id": hr_id, "lane": n["lane"],
                               "name": n["name"] + " (human review)",
                               "human_effort_factor": PARTIAL_RETAIN, "role": "human"})
            id_map[n["id"]] = [ai_id, hr_id]
        else:
            nid = n["id"] + "_m"
            tobe_nodes.append({**base, "id": nid, "lane": n["lane"],
                               "human_effort_factor": 1.0, "role": "human"})
            id_map[n["id"]] = [nid]

    # --- remap edges through the TO-BE id expansion ---
    def entry(s): return id_map[s][0]
    def exit_(s): return id_map[s][-1]
    tobe_edges = [{"from": exit_(e["from"]), "to": entry(e["to"]),
                   "label": e.get("label", "")}
                  for e in edges if e["from"] in id_map and e["to"] in id_map]
    for src, ids in id_map.items():
        if len(ids) == 2:
            tobe_edges.append({"from": ids[0], "to": ids[1], "label": ""})

    lane_order = []
    for u in units:
        a = u.get("actor", MISSING)
        if a not in lane_order:
            lane_order.append(a)

    asis_human = [n for n in asis_nodes if not is_ai(n["lane"])]
    tobe_human = [n for n in tobe_nodes if n["role"] == "human"]
    asis_human_full = [n for n in asis_human if n.get("grade") == "full"]
    asis_human_partial = [n for n in asis_human if n.get("grade") == "partial"]
    node_reduction = round(
        (1 - len(tobe_human) / max(len(asis_human), 1)) * 100, 1)
    return {
        "agent": "agent1_5_process_graph",
        "params": {"partial_retain": PARTIAL_RETAIN, "ai_lane": AI_LANE},
        "lane_order": lane_order,
        "as_is": {"nodes": asis_nodes, "edges": edges},
        "to_be": {"nodes": tobe_nodes, "edges": tobe_edges},
        "counts": {
            # Same schema as the TCB graph so all four cases are interchangeable.
            "asis_total": len(asis_nodes), "asis_human": len(asis_human),
            "tobe_total": len(tobe_nodes), "tobe_human": len(tobe_human),
            "node_count_reduction_pct": node_reduction,          # Metric A
            "fully_eliminated_human": len(asis_human_full),      # Metric C
            "partially_reduced_human": len(asis_human_partial),  # Metric C
            # Metric B (time-weighted) requires Agent 2; these cases stopped at
            # Agent 1.5, so time fields are null — swimlane rendering does not
            # need them, and this keeps the schema aligned with TCB.
            "effort_reduction_pct": None,
            "asis_human_minutes_per_month": None,
            "tobe_human_minutes_per_month": None,
        },
    }

# --- Run for each cross-domain case and save with the TCB-compatible name -----
for domain, units in results.items():
    graph = build_graph_for_case(units)
    out = INFER / f"process_graph_{domain}.json"
    out.write_text(json.dumps(graph, ensure_ascii=False, indent=2), encoding="utf-8")
    c = graph["counts"]
    print(f"[INFO] {domain:26s}: graph -> {rel(out)}  "
          f"(AS-IS human {c['asis_human']} -> TO-BE human {c['tobe_human']})")

print("[INFO] All three cross-domain process graphs generated with the "
      "TCB-compatible schema. TCB's own graph is at process_graph.json.")

[INFO] hospital_billing          : graph -> artifacts\inference\process_graph_hospital_billing.json  (AS-IS human 6 -> TO-BE human 4)
[INFO] law_contract_review       : graph -> artifacts\inference\process_graph_law_contract_review.json  (AS-IS human 7 -> TO-BE human 5)
[INFO] manufacturing_settlement  : graph -> artifacts\inference\process_graph_manufacturing_settlement.json  (AS-IS human 7 -> TO-BE human 5)
[INFO] All three cross-domain process graphs generated with the TCB-compatible schema. TCB's own graph is at process_graph.json.
